# LangGraph - A LangChain-Native ReAct Agent

In this chapter, we will implement a ReAct agent using only `LangChain`, without utilizing `LangGraph`.

Before exploring `LangGraph` in detail, it is useful to review some `LangChain` basics, as `LangGraph`is build upon `LangChain`.

We will create a simple ReAct agent using LangChain.

First, we'll configure API keys for OpenAI and Tavily Search as environment variables.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

Next, we'll import the required modules and setup the language model and tools.

In [2]:
import datetime
from langchain_openai import ChatOpenAI
from langchain_classic.agents import initialize_agent, tool
from langchain_community.tools import TavilySearchResults

A **tool** is like a superpower we give our AI agent. By default, a language model can only answer based on what it knows up until its training date. But with tools, we can extend its abilities — for example, by letting it:

* Search the web for real-time info

* Check the current date and time
* Fetch data from our own database
* And much more!

In [3]:
search_tool = TavilySearchResults(max_results=10)

/tmp/ipykernel_417342/2533520296.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=10)


This tool lets the agent look up information on the web using Tavily Search. It acts like the agent is opening a browser and Googling for us!

Here, `max_results=10` just means: return up to 10 results per search.

In [4]:
@tool
def get_system_time(format: str = "%Y-%m-%d %H:%M:%S"):
    """Returns the current date and time in the specified format"""
    current_time = datetime.datetime.now()
    formatted_time = current_time.strftime(format)
    return formatted_time

This is a custom tool (a Python function) that gives the current date and time. This way, the agent can always know the exact moment when we run our query.

Now we build the agent, giving it:

* The language model

* The list of tools
* Some settings (like which style of agent to use, and whether to print out its thought process)

In [5]:
llm = ChatOpenAI(model="gpt-4o")

tools = [search_tool, get_system_time]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description", # This means the agent will choose which tool to 
                                         # use, on the fly, as it "reacts" to the question
    verbose=True # If True, you'll see its reasoning step by step in the output
)

/tmp/ipykernel_417342/2264645000.py:5: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(


Now, we give the agent a challenging question. The agent will:

1. **Search the web** for the NBA 2025 season winner and when the final game was.
1. **Check the system time**.
1. **Calculate** how many days have passed since that final game.

In [6]:
agent.invoke(
    "Who became the winner of 2025 NBA season? "
    "When did the final game take place and how many days ago was that from this instant?"
)



> Entering new AgentExecutor chain...
To answer the question, I need to first determine the winner of the 2025 NBA season and the date of the final game. Then, I will calculate how many days ago that was from the current date. I'll start by searching for the winner and the date of the final game.

Action: tavily_search_results_json
Action Input: "2025 NBA season winner and final game date"
Observation: [{'title': '2025–26 NBA season', 'url': 'https://en.wikipedia.org/wiki/2025%E2%80%9326_NBA_season', 'content': 'Regular season\n\n[edit]\n\nThe NBA released the regular season schedule on August 14, with select games announced in advance of the full schedule release.(\n\n### International games\n\n[edit]\n\nMain article: NBA Global Games\n\n| Date | Teams | Arena | Location | Reference | Winner |\n ---  ---  --- |\n| NBA Mexico City Game 2025 |\n| November 1 | Detroit Pistons vs. Dallas Mavericks | Mexico City Arena | Mexico City, Mexico | ( | Detroit Pistons |\n| NBA Berlin Game 2026 

{'input': 'Who became the winner of 2025 NBA season? When did the final game take place and how many days ago was that from this instant?',
 'output': 'The Oklahoma City Thunder won the 2025 NBA season. The final game took place on June 22, 2025, and it was 271 days ago from today, March 19, 2026.'}

The agent (powered by GPT-4o) read our question and **broke it into parts**:

* Figure out who won.

* Find the date of the final game.
* Do some date math to calculate “how many days ago.”

**Tool Used**: `tavily search`

* The agent realized it needed *real-time info* (who won the 2025 NBA Finals, when the last game happened).

* It used the Tavily Search tool — think of this as the agent “Googling” for us!
* It searched: “2025 NBA season winner and final game date.”
* From the search results, it found: The **Oklahoma City Thunder** beat the Indiana Pacers in **Game 7**.


**Tool Used**: `tavily_search` (again!)

* The agent knew it needed the *exact date* for Game 7.

* It searched: “2025 NBA Finals Game 7 date.”
* It found: **Game 7 was on June 22, 2025**.

**Tool Used**: `get_system_time`

* The agent now had the final game’s date.

* Next, it needed to know **“what is the current date and time?”**
* It called the `get_system_time` tool, which is just a Python function that returns the current date and time, formatted however we want.
* It got back: **2025–03–19**

Now the agent had all the pieces:

* **Final game**: June 22, 2025

* **Today**: March 19, 2026
* It simply subtracted the two dates and got **271 days**.

The agent put it all together in a nice, complete answer:

***The Oklahoma City Thunder won the 2025 NBA season. The final game took place on June 22, 2025, and it was 271 days ago from today, March 19, 2026.***

The agent reads the question and *decides*, step by step, “What do I need to answer this?” Whenever it needs information outside of its own memory (like real-world facts or the current time), it picks the right tool to help.
